# Registers, Conventions and Oracles
Chapter 7 companion. Use |x1 x0 y⟩=|q2 q1 q0⟩ for two input bits. Truth tables are in ascending input order. Statevectors are ideal model information, not one-shot measurement outputs.


In [ ]:
%pip -q install "qiskit>=2,<3" "numpy>=2,<3"


In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator
z=np.array([1,0],complex)
o=np.array([0,1],complex)
plus=np.array([1,1],complex)/np.sqrt(2)
minus=np.array([1,-1],complex)/np.sqrt(2)
H=np.array([[1,1],[1,-1]],complex)/np.sqrt(2)


## 1. Register labels and asymmetric ordering tests
Drawing order does not change gate action. Classical strings require a separate measurement map. Here the count-layout examples use one classical register.


In [ ]:
qc=QuantumCircuit(3)
qc.x(2)
qc.x(0)
sv=Statevector.from_instruction(qc)
assert np.argmax(abs(sv.data))==5
assert np.isclose(sv.probabilities()[5],1)
print(qc.draw(output="text"))
print(qc.draw(output="text",reverse_bits=True))
assert np.allclose(Statevector.from_instruction(qc).data,sv.data)
quantum_label="100"
print("Identity qi->ci record:",quantum_label)
print("Reversed measurement-wiring record:",quantum_label[::-1])
# QuantumCircuit stores the measurement destinations explicitly.
measured=QuantumCircuit(3,3)
measured.x(2)
measured.measure([2,1,0],[0,1,2])
assert [measured.find_bit(item.qubits[0]).index for item in measured.data if item.operation.name=="measure"]==[2,1,0]
assert [measured.find_bit(item.clbits[0]).index for item in measured.data if item.operation.name=="measure"]==[0,1,2]


## 2. A complete XOR-oracle matrix and a gate implementation
NumPy constructs the basis permutation directly. Qiskit constructs it independently from controlled target flips on each truth-table minterm. The small example is educational, not an efficient general synthesis algorithm.


In [ ]:
def oracle_matrix(f):
    m=len(f)
    assert m in [2,4] and all(b in [0,1] for b in f)
    U=np.zeros((2*m,2*m),complex)
    for x in range(m):
        for y in [0,1]:
            U[2*x+(y^f[x]),2*x+y]=1
    return U
def oracle_circuit(f):
    n=int(np.log2(len(f)))
    qc=QuantumCircuit(n+1,name="Uf")
    for x,fx in enumerate(f):
        if not fx:
            continue
        # input bit j is on qubit j+1; Y is q0
        zeros=[j+1 for j in range(n) if ((x>>j)&1)==0]
        for j in zeros:
            qc.x(j)
        if n==1:
            qc.cx(1,0)
        else:
            qc.ccx(1,2,0)
        for j in reversed(zeros):
            qc.x(j)
    return qc
for n in [1,2]:
    m=2**n
    for mask in range(2**m):
        f=[(mask>>x)&1 for x in range(m)]
        U=oracle_matrix(f)
        assert np.allclose(U.conj().T@U,np.eye(2*m))
        assert np.allclose(U@U,np.eye(2*m))
        assert np.allclose(Operator(oracle_circuit(f)).data,U)
print("All 4 one-input and 16 two-input Boolean oracle matrices agree.")


## 3. Reversible AND and clean uncomputation
Use |x1 x0 T Y⟩=|q3 q2 q1 q0⟩. T is temporary output, Y the useful target. Compute AND into T, toggle Y, then reverse the computation.


In [ ]:
qc=QuantumCircuit(4)
qc.h(3)
qc.h(2)
qc.x(0)
qc.h(0) # Y=minus
qc.ccx(3,2,1) # compute f into T
qc.cx(1,0) # toggle useful target
qc.ccx(3,2,1) # uncompute T
sv=Statevector.from_instruction(qc)
signs=np.array([1,1,1,-1])/2
expected=np.kron(np.kron(signs,z),minus)
assert np.allclose(sv.data,expected)
assert np.allclose(sv.probabilities([1]),[1,0])
print(qc.draw(output="text"))


## 4. One superposed oracle query
The uniform parity example has four nonzero coherent components. One Z sample supplies only one classical label. Each sample_counts trial represents a fresh preparation; the stored statevector does not collapse.


In [ ]:
parity=[0,1,1,0]
input_state=np.kron(np.ones(4)/2,z)
sv=Statevector(input_state).evolve(oracle_circuit(parity))
expected=oracle_matrix(parity)@input_state
assert np.allclose(sv.data,expected)
assert np.allclose(sv.probabilities(),[.25,0,0,.25,0,.25,.25,0])
sv.seed(2026)
saved=sv.data.copy()
print("One fresh Z record:",sv.sample_counts(1))
print("200 fresh Z records:",sv.sample_counts(200))
assert np.allclose(sv.data,saved)


## 5. Phase kickback for every small truth table
Test arbitrary complex input amplitudes, not only a uniform input. The target factors out as minus while each input coefficient gains (-1)^f(x). A plus target gives no function-dependent phase.


In [ ]:
rng=np.random.default_rng(2026)
for n in [1,2]:
    m=2**n
    alpha=rng.normal(size=m)+1j*rng.normal(size=m)
    alpha/=np.linalg.norm(alpha)
    for mask in range(2**m):
        f=np.array([(mask>>x)&1 for x in range(m)])
        U=oracle_matrix(f)
        assert np.allclose(U@np.kron(alpha,minus),np.kron(alpha*((-1)**f),minus))
        assert np.allclose(U@np.kron(alpha,plus),np.kron(alpha,plus))


## 6. Recombining the signs
For parity, H on both input qubits produces input 11. For AND, final input probabilities are uniform. These examples do not imply a universal rule for learning any truth table in one shot.


In [ ]:
for f in [[0,1,1,0],[0,0,0,1],[0,0,0,0],[1,1,1,1]]:
    qc=QuantumCircuit(3)
    qc.h(2)
    qc.h(1)
    qc.x(0)
    qc.h(0)
    qc.compose(oracle_circuit(f),inplace=True)
    before=Statevector.from_instruction(qc)
    qc.h(2)
    qc.h(1)
    after=Statevector.from_instruction(qc)
    expected_input=np.kron(H,H)@(np.ones(4)/2*((-1)**np.array(f)))
    assert np.allclose(after.data,np.kron(expected_input,minus))
    input_probs=np.abs(after.data.reshape(4,2))**2
    input_probs=input_probs.sum(axis=1)
    print(f,"input Z probabilities:",input_probs)
    if f==[0,1,1,0]:
        assert np.allclose(input_probs,[0,0,0,1])
    elif f==[0,0,0,1]:
        assert np.allclose(input_probs,[.25]*4)


## Practice
1. Change the input weights and predict which basis pairs appear for a zero target.
2. Derive the oracle for one-input truth table 10, including both target-input values.
3. Track the common sign for constant-one kickback and explain why it does not change readout.
4. Explain the distinction between one oracle query and the gate count of its implementation.

References: [Qiskit bit ordering](https://quantum.cloud.ibm.com/docs/en/guides/bit-ordering), [QuantumCircuit API](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.QuantumCircuit).
